# Stage A - the sparse substrate

**Question: what is the best sparse-training substrate, before BaCP is involved at all?**
EAST is baseline machinery here, not the contribution.

| Rung | Change | Answers |
|---|---|---|
| A0 | Dense | The ceiling, and the checkpoint every sparse rung starts from |
| A1 | Uniform magnitude, head pruned | The floor (optional; cut first if budget is tight) |
| A2 | + head dense | How much of "the ERK effect" is just protecting the classifier |
| A3 | ERK, head pruned | ERK's allocation principle alone |
| A4 | ERK + head dense | Do they compose, or is ERK approximately head protection? |
| A5 | + RigL drop-and-regrow | Does dynamic topology add beyond allocation? |
| A6 | + EAST cyclic sparsity | EAST's schedule vs RigL's |

A1-A4 form a deliberate **2x2** on {uniform, ERK} x {head pruned, head dense}.
Collapsing it into one rung would permanently confound "ERK helps" with "ERK
protects the classifier" - and at 99.9% the classifier is ~24% of the entire
surviving weight budget, so that confound is not hypothetical.

The main path runs through A3/A5, the head-**pruned** corner, because that is what
RigL and GraNet actually do. A2/A4 measure what the other convention buys rather
than adopting it.

**A0 must complete before anything else**: every sparse rung resolves a dense
checkpoint *of the same seed* at execution time.

> Unpaired: the mask lottery dominates run-to-run variance here, so seed correlation between arms is low and pairing buys little.


In [ ]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd()))          # so ladder_nb is importable
import ladder_nb as nb
info = nb.setup()


## Configure

`TIER` 1 is the spine (5 seeds). `PER_GPU` 2 saturates a card at this model size - one cell per GPU leaves it at ~53% utilisation, and each cell only needs about 815 MB.


In [ ]:
TIER    = 1
GPUS    = info['gpus'] or 1
PER_GPU = 2
SEEDS   = None          # None = every seed the tier schedules

RUNGS = ['A0', 'A2', 'A3', 'A4', 'A5', 'A6']

import manifest as M
grid = [c for c in M.cells(TIER, rungs=RUNGS)]
print(f'{len(grid)} cell(s) planned over {len(set(c["rung"] for c in grid))} rung(s)')
for r in RUNGS:
    n = sum(1 for c in grid if c['rung'] == r)
    print(f'  {r:10s} {n} seed(s)' if n else f'  {r:10s} -- NOT IN TIER {TIER}')


## Run

Idempotent - a cell is complete iff a record carrying its key exists, so re-running skips what is done. Dense cells run first as a hard barrier. Safe to interrupt; you lose at most the cells in flight.


In [ ]:
summary = nb.run_stage(RUNGS, tier=TIER, gpus=GPUS, per_gpu=PER_GPU, seeds=SEEDS)
print(summary['ok'], 'ok,', summary['failed'], 'failed,', summary['skipped'], 'skipped')


## Progress and health

The `nan` column is the one to read first. A diverged run sits at exactly 10.0969% (chance on CIFAR-10) for the rest of training and every delta computed from it is meaningless.


In [ ]:
nb.progress(TIER)


## Watch a single cell

Use this when something looks wrong - it streams one line per epoch so you can see *where* a run breaks rather than only that it did.


In [ ]:
cell = nb.attach(nb.pick('A0', seed=1, tier=TIER))
nb.show(cell)
# hist, first_nan = nb.watch(cell, gpu=0)
# nb.plot(hist, first_nan, cell['key'])


## Table and gates


In [ ]:
out = nb.report(TIER)
